# PGVector RAG Query Notebook — Keycloak JWT + RLS Demo

This notebook demonstrates role-scoped document access using PGVector + LangChain, governed by **Keycloak JWTs** — the same identity model used by the MCP server.

- **Jane** (admin token) can query documents from **all** collections
- **John** (user token) can only query documents from the **`user`** collection

The Keycloak JWT determines `app.current_role`, which PostgreSQL RLS policies use to filter rows.

In [ ]:
import base64
import json
import os
from urllib.parse import quote

import requests
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGEngine, PGVectorStore
import psycopg

# PostgreSQL connection (same credentials as the MCP server)
PG_HOST = os.getenv("PG_HOST", "postgresql.redbank-demo.svc.cluster.local")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DATABASE = os.getenv("PG_DATABASE", "db")
PG_USER = os.getenv("PG_USER", "user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "pass")

# Keycloak configuration
KEYCLOAK_URL = os.getenv("KEYCLOAK_URL", "https://keycloak-keycloak.apps.redbank-demo2.ij5f.p3.openshiftapps.com")
KEYCLOAK_REALM = os.getenv("KEYCLOAK_REALM", "redbank")
KEYCLOAK_CLIENT = os.getenv("KEYCLOAK_CLIENT", "redbank-mcp")
ADMIN_ROLE = "admin"

# Embedding model — same as the ingestion pipeline
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5")

print("Setup complete")

## Keycloak Authentication

Get JWTs for Jane (admin) and John (user), then extract their roles from the token claims — the same way the MCP server does it.

In [ ]:
def get_keycloak_token(username: str, password: str) -> str:
    """Get an access token from Keycloak."""
    resp = requests.post(
        f"{KEYCLOAK_URL}/realms/{KEYCLOAK_REALM}/protocol/openid-connect/token",
        data={
            "grant_type": "password",
            "client_id": KEYCLOAK_CLIENT,
            "username": username,
            "password": password,
        },
        verify=False,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def extract_role(token: str) -> str:
    """Extract role from JWT claims (same logic as MCP server)."""
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # pad base64
    claims = json.loads(base64.urlsafe_b64decode(payload))
    # Check realm_access.roles for admin role
    realm_roles = claims.get("realm_access", {}).get("roles", [])
    return "admin" if ADMIN_ROLE in realm_roles else "user"


def build_connection_string(role: str) -> str:
    """Build a PostgreSQL connection string that sets app.current_role via options."""
    return (
        f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}"
        f"@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
        f"?options={quote(f'-c app.current_role={role}')}"
    )


# Authenticate as Jane (admin) and John (user)
jane_token = get_keycloak_token("jane", "jane123")
jane_role = extract_role(jane_token)
print(f"Jane: role={jane_role}")

john_token = get_keycloak_token("john", "john123")
john_role = extract_role(john_token)
print(f"John: role={john_role}")

## Jane's Query (Admin) — sees all collections

Jane's JWT contains the `admin` realm role, so `app.current_role='admin'` is set on the connection. RLS allows her to see both `admin` and `user` documents.

In [ ]:
jane_engine = PGEngine.from_connection_string(url=build_connection_string(jane_role))

jane_store = PGVectorStore.create_sync(
    engine=jane_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

query = "How do I reset my password?"
jane_results = jane_store.similarity_search(query, k=5)

print(f"Jane (role={jane_role}) results ({len(jane_results)} docs):\n")
for i, doc in enumerate(jane_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

## John's Query (User) — sees only `user` collection

John's JWT has no `admin` role, so `app.current_role='user'` is set. RLS restricts him to the `user` collection only — he cannot see admin-only internal procedures.

In [ ]:
john_engine = PGEngine.from_connection_string(url=build_connection_string(john_role))

john_store = PGVectorStore.create_sync(
    engine=john_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

john_results = john_store.similarity_search(query, k=5)

print(f"John (role={john_role}) results ({len(john_results)} docs):\n")
for i, doc in enumerate(john_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

## RLS Proof — side-by-side comparison

Compare results and verify via direct SQL that `app.current_role='user'` cannot see admin rows.

In [ ]:
print("=== RLS Proof ===\n")
print(f"Jane ({jane_role}) saw {len(jane_results)} results")
jane_collections = set(doc.metadata.get("collection") for doc in jane_results)
print(f"  Collections: {sorted(jane_collections)}")

print(f"\nJohn ({john_role}) saw {len(john_results)} results")
john_collections = set(doc.metadata.get("collection") for doc in john_results)
print(f"  Collections: {sorted(john_collections)}")

# Direct SQL proof
print("\n--- Direct SQL verification ---")

with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DATABASE} "
    f"user={PG_USER} password={PG_PASSWORD}"
) as conn:
    conn.execute("SELECT set_config('app.current_role', 'user', false)")
    row = conn.execute(
        "SELECT count(*) FROM embeddings WHERE collection = 'admin'"
    ).fetchone()
    print(f"app.current_role='user'  -> admin rows visible: {row[0]} (expected 0)")

    conn.execute("SELECT set_config('app.current_role', 'admin', false)")
    row = conn.execute(
        "SELECT count(*) FROM embeddings WHERE collection = 'admin'"
    ).fetchone()
    print(f"app.current_role='admin' -> admin rows visible: {row[0]}")

## Document Count per Collection

In [ ]:
with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DATABASE} "
    f"user={PG_USER} password={PG_PASSWORD}"
) as conn:
    conn.execute("SELECT set_config('app.current_role', 'admin', false)")
    rows = conn.execute(
        "SELECT collection, count(*) FROM embeddings GROUP BY collection ORDER BY collection"
    ).fetchall()

print("Document chunks per collection:")
for collection, count in rows:
    print(f"  {collection}: {count}")